# Setup: Synthetic Data, Model Training & Registration

This notebook creates the shared foundation for all monitoring demos. Run it once before any of the other notebooks.

**What this notebook does:**

1. Creates Snowflake infrastructure (database, schema, warehouse, tables)
2. Generates synthetic customer churn data (5,000 rows)
3. Stores train/test splits as Snowflake tables (`CHURN_TRAIN`, `CHURN_TEST`)
4. Trains two models:
   - **V1** -- Logistic Regression (baseline)
   - **V2** -- XGBoost (challenger)
5. Registers both as `CHURN_MODEL` V1 and V2 in the Snowflake Model Registry

**After running this notebook**, you can jump into any demo notebook:
- `01_model_monitor.ipynb` -- Model version monitoring
- `02_model_monitor_custom.ipynb` -- Custom metric monitoring
- `03_gateway_monitor.ipynb` -- Gateway A/B testing with traffic splits

## 1. Imports

Import the libraries used throughout this setup: numpy/pandas for data generation, scikit-learn and xgboost for model training, and Snowpark + Model Registry for Snowflake integration.

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

## 2. Connect to Snowflake

**Snowsight (Workspaces):** The session is pre-authenticated -- no configuration needed. The cell below auto-detects this via `get_active_session()`.

**Local development:** Falls back to keypair authentication. Update the account, user, and key path in the `LOCAL CONFIG` section below to match your environment.

In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from cryptography.hazmat.primitives import serialization
    from pathlib import Path

    # ── LOCAL CONFIG (update these for your environment) ──
    ACCOUNT = "<your-account-identifier>"
    USER = "<your-username>"
    ROLE = "ACCOUNTADMIN"
    KEY_PATH = Path.home() / ".snowflake" / "keys" / "rsa_key.p8"
    # ──────────────────────────────────────────────────────

    with open(KEY_PATH, "rb") as f:
        private_key = serialization.load_pem_private_key(f.read(), password=None)

    private_key_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )

    session = Session.builder.configs({
        "account": ACCOUNT,
        "user": USER,
        "private_key": private_key_bytes,
        "role": ROLE,
        "database": "ML_DEMO",
        "schema": "ML_CHURN",
        "warehouse": "ML_CHURN_WH"
    }).create()

print(f"Connected as: {session.get_current_role()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

## 3. Create Infrastructure

Creates the database, schema, warehouse, and grants needed by all demo notebooks. Safe to re-run (uses `IF NOT EXISTS`).

In [ ]:
session.sql("CREATE DATABASE IF NOT EXISTS ML_DEMO").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS ML_DEMO.ML_CHURN").collect()
session.sql("""
CREATE WAREHOUSE IF NOT EXISTS ML_CHURN_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
""").collect()

session.sql("""
CREATE TABLE IF NOT EXISTS ML_DEMO.ML_CHURN.GROUND_TRUTH (
    "request_id" VARCHAR,
    "churned" NUMBER
)
""").collect()

session.sql("GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE ACCOUNTADMIN").collect()
session.sql("GRANT CREATE GATEWAY ON SCHEMA ML_DEMO.ML_CHURN TO ROLE ACCOUNTADMIN").collect()
session.sql("GRANT CREATE MODEL MONITOR ON SCHEMA ML_DEMO.ML_CHURN TO ROLE ACCOUNTADMIN").collect()

print("Infrastructure ready.")

## 4. Generate Synthetic Churn Data

Create a 5,000-row synthetic customer dataset with 6 features and a binary churn target. The target is generated via a logistic function with realistic signal from tenure, charges, support tickets, and contract type. A fixed random seed (42) ensures reproducibility across runs.

In [ ]:
np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    "TENURE_MONTHS": np.random.randint(1, 72, n_samples),
    "MONTHLY_CHARGES": np.round(np.random.uniform(20, 120, n_samples), 2),
    "TOTAL_CHARGES": np.round(np.random.uniform(100, 8000, n_samples), 2),
    "CONTRACT_TYPE": np.random.choice([0, 1, 2], n_samples, p=[0.5, 0.3, 0.2]),
    "NUM_SUPPORT_TICKETS": np.random.poisson(2, n_samples),
    "INTERNET_SERVICE": np.random.choice([0, 1, 2], n_samples, p=[0.2, 0.4, 0.4]),
})

churn_prob = (
    -0.02 * data["TENURE_MONTHS"]
    + 0.01 * data["MONTHLY_CHARGES"]
    + 0.15 * data["NUM_SUPPORT_TICKETS"]
    - 0.5 * data["CONTRACT_TYPE"]
    + np.random.normal(0, 0.5, n_samples)
)
data["CHURNED"] = (expit(churn_prob) > 0.5).astype(int)

print(f"Dataset shape: {data.shape}")
print(f"Churn rate: {data['CHURNED'].mean():.2%}")
data.head()

## 5. Train/Test Split and Store to Snowflake

Stores both splits as Snowflake tables so all downstream notebooks share the exact same data.

In [ ]:
FEATURE_COLS = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
                "CONTRACT_TYPE", "NUM_SUPPORT_TICKETS", "INTERNET_SERVICE"]

X = data[FEATURE_COLS]
y = data["CHURNED"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = X_train.copy()
train_df["CHURNED"] = y_train.values

test_df = X_test.copy()
test_df["CHURNED"] = y_test.values

session.write_pandas(train_df, table_name="CHURN_TRAIN", database="ML_DEMO",
                     schema="ML_CHURN", overwrite=True, auto_create_table=True)
session.write_pandas(test_df, table_name="CHURN_TEST", database="ML_DEMO",
                     schema="ML_CHURN", overwrite=True, auto_create_table=True)

print(f"Train: {len(train_df)} rows -> ML_DEMO.ML_CHURN.CHURN_TRAIN")
print(f"Test:  {len(test_df)} rows -> ML_DEMO.ML_CHURN.CHURN_TEST")

## 6. Train Model V1: Logistic Regression (Baseline)

Train a simple logistic regression as the baseline model. This serves as the "incumbent" in A/B testing scenarios (notebook 03) and the primary model for monitoring demos (notebooks 01 and 02).

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
print("=== Model V1: Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_lr):.4f}")

## 7. Train Model V2: XGBoost (Challenger)

Train an XGBoost classifier as the challenger model. In the gateway A/B testing notebook (03), this competes against V1 with traffic split between both. Both models are trained on the same data for a fair comparison.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print("=== Model V2: XGBoost ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_xgb):.4f}")

## 8. Register Both Models in Snowflake Model Registry

Log both trained models to the Snowflake Model Registry as `CHURN_MODEL` V1 (LogReg) and V2 (XGBoost). Registration packages the model artifact, dependencies, and sample input schema so the model can be deployed for inference or monitored from any downstream notebook. If re-running, the existing model is dropped first.

In [ ]:
reg = Registry(session=session)

# Drop existing model if re-running setup
session.sql("DROP MODEL IF EXISTS ML_DEMO.ML_CHURN.CHURN_MODEL").collect()

mv_v1 = reg.log_model(
    model=lr_model,
    model_name="CHURN_MODEL",
    version_name="V1",
    sample_input_data=X_test.head(5),
    conda_dependencies=["scikit-learn"],
    comment="Logistic Regression baseline for churn prediction"
)
print("Registered CHURN_MODEL V1 (LogisticRegression)")

mv_v2 = reg.log_model(
    model=xgb_model,
    model_name="CHURN_MODEL",
    version_name="V2",
    sample_input_data=X_test.head(5),
    conda_dependencies=["xgboost"],
    comment="XGBoost challenger for churn prediction"
)
print("Registered CHURN_MODEL V2 (XGBoost)")

## 9. Verify

Confirm both model versions are registered and accessible. After this cell succeeds, you can close this notebook and open any of the demo notebooks (01, 02, or 03).

In [ ]:
model = reg.get_model("CHURN_MODEL")
versions = model.show_versions()
print("Registered model versions:")
print(versions[["name", "comment", "created_on"]])

print("\n--- Setup complete ---")
print("You can now run any of the monitoring demo notebooks.")

In [ ]:
session.close()
print("Session closed.")